## **Hyperparameter Tuning using Optuna**

### **Roadmap**

**1. Setup and Data Preparation**

**2. Single Model Optimization**

**3. Multiple Model Optimization**

**4. Optuna Visualizations**

**5. Pruning with XGBoost**

### **Setup and Data Preparation**

Install the required libraries and load the dataset.

Imputing missing values and scaling numerical features ensures the model weights compute correctly.

In [24]:
import numpy as np
import pandas as pd
import optuna
import xgboost as xgb
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from sklearn.datasets import load_iris
import warnings
warnings.filterwarnings("ignore")

In [25]:
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
columns = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome']
diabetes_df = pd.read_csv(url, names=columns)

cols_with_missing_vals = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
diabetes_df[cols_with_missing_vals] = diabetes_df[cols_with_missing_vals].replace(0, np.nan)
diabetes_df.fillna(diabetes_df.mean(), inplace=True)

X_diabetes = diabetes_df.drop('Outcome', axis=1)
y_diabetes = diabetes_df['Outcome']

X_train_db, X_test_db, y_train_db, y_test_db = train_test_split(X_diabetes, y_diabetes, test_size=0.3, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_db)
X_test_scaled = scaler.transform(X_test_db)

### **Single Model Optimization**

Define an objective function to tune hyperparameters for a specific algorithm.

Optuna utilizes the Tree-structured Parzen Estimator (TPE) algorithm by default to find optimal parameters efficiently.

In [ ]:
def rf_objective(trial):
    n_estimators = trial.suggest_int('n_estimators', 50, 200)
    max_depth = trial.suggest_int('max_depth', 3, 20)

    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=42
    )

    score = cross_val_score(model, X_train_scaled, y_train_db, cv=3, scoring='accuracy').mean()
    return score

rf_study = optuna.create_study(direction='maximize')
rf_study.optimize(rf_objective, n_trials=10)

print(f"Best hyperparameters: {rf_study.best_trial.params}")

### **Multiple Model Optimization**

Select the best algorithm alongside its optimal hyperparameters dynamically.

The `suggest_categorical` method dictates which classifier branch executes during a specific trial.

In [ ]:
def multi_model_objective(trial):
    classifier_name = trial.suggest_categorical('classifier', ['SVM', 'RandomForest'])

    if classifier_name == 'SVM':
        c_val = trial.suggest_float('C', 0.1, 10.0, log=True)
        model = SVC(C=c_val, random_state=42)
    else:
        n_estimators = trial.suggest_int('n_estimators', 50, 150)
        model = RandomForestClassifier(n_estimators=n_estimators, random_state=42)

    score = cross_val_score(model, X_train_scaled, y_train_db, cv=3, scoring='accuracy').mean()
    return score

multi_study = optuna.create_study(direction='maximize')
multi_study.optimize(multi_model_objective, n_trials=10)

print(f"Best model and parameters: {multi_study.best_trial.params}")

### **Optuna Visualizations**

Generate plots to analyze hyperparameter importance and trial history.

Visualizations map out parameter correlations and the overall optimization progress.

In [28]:
from optuna.visualization import plot_optimization_history, plot_param_importances

plot_optimization_history(rf_study).show()
plot_param_importances(rf_study).show()

### **Pruning with XGBoost**

Halt unpromising trials early using a pruner.

The `SuccessiveHalvingPruner` terminates poor-performing configurations to conserve computational resources.

In [ ]:
X_iris, y_iris = load_iris(return_X_y=True)
X_train_iris, X_test_iris, y_train_iris, y_test_iris = train_test_split(X_iris, y_iris, test_size=0.2, random_state=42)

def xgb_objective(trial):
    param = {
        'verbosity': 0,
        'objective': 'multi:softprob',
        'num_class': 3,
        'eval_metric': 'mlogloss',
        'lambda': trial.suggest_float('lambda', 1e-3, 1.0, log=True),
        'alpha': trial.suggest_float('alpha', 1e-3, 1.0, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 9)
    }

    dtrain = xgb.DMatrix(X_train_iris, label=y_train_iris)
    dtest = xgb.DMatrix(X_test_iris, label=y_test_iris)

    pruning_callback = optuna.integration.XGBoostPruningCallback(trial, "eval-mlogloss")

    bst = xgb.train(
        param,
        dtrain,
        num_boost_round=50,
        evals=[(dtrain, "train"), (dtest, "eval")],
        early_stopping_rounds=10,
        callbacks=[pruning_callback]
    )

    preds = bst.predict(dtest)
    best_preds = [int(np.argmax(line)) for line in preds]
    return accuracy_score(y_test_iris, best_preds)

xgb_study = optuna.create_study(direction='maximize', pruner=optuna.pruners.SuccessiveHalvingPruner())
xgb_study.optimize(xgb_objective, n_trials=10)

print(f"Best accuracy: {xgb_study.best_value}")

### **Key Revision Notes**

- Optuna is reliant on an `objective` function that returns a metric to maximize or minimize.
- The `suggest_*` methods directly define the search space limits (e.g., `suggest_int`, `suggest_float`, `suggest_categorical`).
- A `Study` object manages the optimization history and outputs the best parameters.
- Pruners execute early stopping on unpromising trials to reduce total execution time.